# Autoencoders & Latent Spaces

This notebook accompanies the **ML Viz** lesson on autoencoders.
We'll build a vanilla autoencoder from scratch and explore the latent space.

**Companion lesson:** https://ml-viz.vercel.app/courses/generative-models/02-autoencoders

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Autoencoder architecture

An autoencoder has two parts:
- **Encoder**: compresses input $x$ into a latent vector $z$
- **Decoder**: reconstructs $x$ from $z$

We'll implement this using only NumPy — no frameworks needed.

In [ ]:
class Autoencoder:
    """Simple linear autoencoder with ReLU activations."""
    
    def __init__(self, input_dim, latent_dim):
        scale = np.sqrt(2.0 / input_dim)
        self.W_enc = np.random.randn(input_dim, latent_dim) * scale
        self.b_enc = np.zeros(latent_dim)
        self.W_dec = np.random.randn(latent_dim, input_dim) * scale
        self.b_dec = np.zeros(input_dim)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def encode(self, x):
        self.z_pre = x @ self.W_enc + self.b_enc
        self.z = self.relu(self.z_pre)
        return self.z
    
    def decode(self, z):
        self.x_hat = z @ self.W_dec + self.b_dec
        return self.x_hat
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
    
    def loss(self, x):
        x_hat = self.forward(x)
        return np.mean((x - x_hat) ** 2)
    
    def backward(self, x, lr=0.01):
        n = x.shape[0]
        x_hat = self.forward(x)
        
        # Gradient of MSE loss w.r.t. output
        dx_hat = 2 * (x_hat - x) / n
        
        # Decoder gradients
        dW_dec = self.z.T @ dx_hat
        db_dec = dx_hat.sum(axis=0)
        
        # Through ReLU
        dz = dx_hat @ self.W_dec.T
        dz_pre = dz * (self.z_pre > 0).astype(float)  # ReLU gradient
        
        # Encoder gradients
        dW_enc = x.T @ dz_pre
        db_enc = dz_pre.sum(axis=0)
        
        # Update
        self.W_enc -= lr * dW_enc
        self.b_enc -= lr * db_enc
        self.W_dec -= lr * dW_dec
        self.b_dec -= lr * db_dec

print('Autoencoder class defined.')

## Generate 2D Swiss Roll data

A Swiss roll is a classic dataset where a 2D manifold is embedded in higher-dimensional space.
The autoencoder should learn to unroll it.

In [ ]:
np.random.seed(42)
n_points = 500

t = 1.5 * np.pi * (1 + 2 * np.random.rand(n_points))
x_swiss = np.column_stack([
    t * np.cos(t),
    t * np.sin(t),
    30 * np.random.rand(n_points)  # noise in 3rd dimension
])

# Normalize
x_swiss = (x_swiss - x_swiss.mean(axis=0)) / x_swiss.std(axis=0)

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x_swiss[:, 0], x_swiss[:, 1], x_swiss[:, 2], c=t, cmap='viridis', s=10, alpha=0.7)
ax.set_title('Swiss Roll Dataset', color='white', fontsize=12)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$x_3$')
ax.view_init(elev=20, azim=45)
plt.tight_layout()
plt.show()

## Train the autoencoder

We'll compress from 3D to 2D (the true intrinsic dimensionality of the Swiss roll).

In [ ]:
ae = Autoencoder(input_dim=3, latent_dim=2)

losses = []
for epoch in range(500):
    loss = ae.loss(x_swiss)
    losses.append(loss)
    ae.backward(x_swiss, lr=0.005)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:3d} | MSE Loss: {loss:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#818cf8', linewidth=1.5)
ax.set_title('Training Loss', color='white', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

## Visualize the latent space

The encoder should map the 3D Swiss roll into a clean 2D representation.

In [ ]:
z_encoded = ae.encode(x_swiss)
x_reconstructed = ae.forward(x_swiss)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Original 3D
ax = fig.add_subplot(131, projection='3d')
ax.scatter(x_swiss[:, 0], x_swiss[:, 1], x_swiss[:, 2], c=t, cmap='viridis', s=8, alpha=0.6)
ax.set_title('Original (3D)', color='white', fontsize=11)
ax.view_init(elev=20, azim=45)

# Latent space
axes[1].scatter(z_encoded[:, 0], z_encoded[:, 1], c=t, cmap='viridis', s=10, alpha=0.7)
axes[1].set_title('Latent Space (2D)', color='white', fontsize=11)
axes[1].set_xlabel('$z_1$')
axes[1].set_ylabel('$z_2$')

# Reconstruction
ax = fig.add_subplot(133, projection='3d')
ax.scatter(x_reconstructed[:, 0], x_reconstructed[:, 1], x_reconstructed[:, 2], c=t, cmap='viridis', s=8, alpha=0.6)
ax.set_title('Reconstruction', color='white', fontsize=11)
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

## The problem: latent space holes

A standard autoencoder maps training inputs to points, but the latent space
between them is unstructured. Sampling randomly produces garbage.

In [ ]:
# Sample random points from latent space
np.random.seed(7)
z_random = np.random.randn(16, 2) * 1.5  # wider than the encoded distribution
x_random = z_random @ ae.W_dec + ae.b_dec  # decode directly

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Random Latent Samples → Decoded (Unstructured = Garbage)', color='white', fontsize=12, y=1.02)

for i in range(16):
    ax = axes[i // 8, i % 8]
    # Show as a simple bar chart of the 3D output
    ax.bar(range(3), x_random[i], color='#f43f5e', alpha=0.8)
    ax.set_ylim(-3, 3)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('These random samples don\'t look like the original data.')
print('The latent space has holes — the decoder was never trained on these regions.')
print('This is why we need VAEs (next lesson) to structure the latent space.')

## Key takeaways

1. Autoencoders learn to compress and reconstruct data
2. The bottleneck forces the network to learn meaningful representations
3. But the latent space is unstructured — random sampling produces garbage
4. **Next lesson:** VAEs fix this by forcing the latent space to follow $\mathcal{N}(0, I)$